# Soil Classification – USCS (Unified Soil Classification System)
## Sieve Analysis + Atterberg Limits

This notebook classifies soil according to the **Unified Soil Classification
System (USCS / ASTM D2487)** using:

1. **Sieve Analysis** – Grain-size distribution curve, D10 / D30 / D60, Cu, Cc
2. **Atterberg Limits** – Liquid Limit (LL), Plastic Limit (PL), Plasticity
   Index (PI)

### USCS Classification Logic
```
Percent fines (passing #200 / 0.075 mm sieve)
├── < 50 %  → COARSE-GRAINED
│   ├── > 50 % retained on #4 (4.75 mm) → GRAVEL (G)
│   │   └── Fines < 5 %  → Cu & Cc → GW / GP
│   │       Fines 5–12 % → Borderline (dual symbol)
│   │       Fines > 12 % → PI & A-line → GM / GC
│   └── ≤ 50 % retained on #4            → SAND (S)
│       └── (same Cu / Cc / PI logic)    → SW / SP / SM / SC
└── ≥ 50 %  → FINE-GRAINED
    └── PI & LL → A-line → CL / CH / ML / MH / OL / OH
```

### How to use
1. Edit `data/soil.xlsx` and `data/atterberg.xlsx` with your measurements.
2. Run all cells (`Runtime → Run all` in Colab, or `Kernel → Restart & Run All` locally).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pd.set_option("display.float_format", "{:.3f}".format)
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})


## 1. Load Input Data

In [ ]:
# ── USER: update paths if needed ─────────────────────────────────────────────
SIEVE_PATH     = "data/soil.xlsx"
ATTERBERG_PATH = "data/atterberg.xlsx"
# ─────────────────────────────────────────────────────────────────────────────

sieve     = pd.read_excel(SIEVE_PATH)
atterberg = pd.read_excel(ATTERBERG_PATH)

print("=== Sieve Analysis Data ===")
print(sieve.to_string(index=False))
print()
print("=== Atterberg Limits Data ===")
print(atterberg.to_string(index=False))


## 2. Sieve Analysis Calculations

Steps:
1. Sum all retained masses → Total mass
2. % Retained = (Mass Retained / Total Mass) × 100
3. Cumulative % Retained = running sum of % Retained
4. % Passing = 100 − Cumulative % Retained


In [ ]:
total_mass = sieve["Mass Retained (g)"].sum()
print(f"Total Sample Mass: {total_mass:.2f} g")

sieve["% Retained"]            = 100 * sieve["Mass Retained (g)"] / total_mass
sieve["Cumul. % Retained"]     = sieve["% Retained"].cumsum()
sieve["Cumul. Mass Retained"]  = sieve["Mass Retained (g)"].cumsum()
sieve["% Passing"]             = 100 * (1 - sieve["Cumul. Mass Retained"] / total_mass)

print()
print("=== Sieve Analysis Summary ===")
print(sieve[["Sieve Size (mm)", "Mass Retained (g)",
             "% Retained", "Cumul. % Retained", "% Passing"]].to_string(index=False))


## 3. Characteristic Grain Diameters (D10, D30, D60)

Determined by **log-linear interpolation** on the grain-size distribution curve:
- **D10** – effective size (10 % passing)
- **D30** – used to compute Cc
- **D60** – controls the uniformity coefficient Cu

$$C_u = \frac{D_{60}}{D_{10}} \qquad C_c = \frac{D_{30}^2}{D_{10} \cdot D_{60}}$$


In [ ]:
# Exclude pan (0 mm) because log(0) is undefined
data   = sieve[sieve["Sieve Size (mm)"] > 0].copy()
sizes   = data["Sieve Size (mm)"].values
passing = data["% Passing"].values

# Sort ascending for interpolation (passing increases as size decreases)
log_sizes = np.log10(sizes)

def find_D(target_pct):
    """Return grain diameter (mm) at which `target_pct` % of soil passes."""
    # Interpolate in log-space; passing array must be monotone
    log_D = np.interp(target_pct, passing[::-1], log_sizes[::-1])
    return 10 ** log_D

D10 = find_D(10)
D30 = find_D(30)
D60 = find_D(60)

Cu = D60 / D10
Cc = (D30 ** 2) / (D10 * D60)

print(f"D10  = {D10:.4f} mm")
print(f"D30  = {D30:.4f} mm")
print(f"D60  = {D60:.4f} mm")
print()
print(f"Coefficient of Uniformity   Cu = D60/D10           = {Cu:.2f}")
print(f"Coefficient of Curvature    Cc = D30²/(D10·D60)    = {Cc:.2f}")


## 4. Grain-Size Distribution Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(data["Sieve Size (mm)"], data["% Passing"],
        marker="o", linestyle="-", color="royalblue",
        linewidth=2, label="Grain-size curve")

# Mark D10, D30, D60
for D, label, color in [(D10, "D₁₀", "red"), (D30, "D₃₀", "orange"), (D60, "D₆₀", "green")]:
    pct = {"D₁₀": 10, "D₃₀": 30, "D₆₀": 60}[label]
    ax.axvline(D, color=color, linestyle="--", linewidth=1.2, alpha=0.8)
    ax.axhline(pct, color=color, linestyle=":", linewidth=1.0, alpha=0.6)
    ax.annotate(f"{label} = {D:.3f} mm",
                xy=(D, pct), xytext=(8, 4), textcoords="offset points",
                color=color, fontsize=9, fontweight="bold")

# USCS grain size boundaries
for x, lbl in [(4.75, "Gravel | Sand"), (0.075, "Sand | Fines")]:
    ax.axvline(x, color="gray", linestyle=":", linewidth=1.2)
    ax.text(x, 102, lbl, ha="center", fontsize=8, color="gray")

ax.set_xscale("log")
ax.set_xlim(left=0.01)
ax.set_ylim(-2, 110)
ax.set_xlabel("Grain Size (mm)", fontsize=12)
ax.set_ylabel("Percent Passing (%)", fontsize=12)
ax.set_title("Grain-Size Distribution Curve", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, which="both", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("grain_size_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → grain_size_curve.png")


## 5. Atterberg Limits

- **Liquid Limit (LL)** – water content at which soil transitions from plastic to liquid state
- **Plastic Limit (PL)** – water content at which soil transitions from semi-solid to plastic state
- **Plasticity Index (PI)** = LL − PL


In [ ]:
LL = atterberg.loc[atterberg["Parameter"] == "Liquid Limit (LL)",  "Value"].values[0]
PL = atterberg.loc[atterberg["Parameter"] == "Plastic Limit (PL)",  "Value"].values[0]
PI = LL - PL

print(f"Liquid Limit  (LL) = {LL}")
print(f"Plastic Limit (PL) = {PL}")
print(f"Plasticity Index (PI) = LL - PL = {PI}")


## 6. Plasticity Chart (Casagrande)

In [ ]:
LL_range = np.linspace(0, 100, 300)
A_line   = 0.73 * (LL_range - 20)          # A-line: PI = 0.73(LL-20)
U_line   = 0.9  * (LL_range - 8)           # U-line: PI = 0.9(LL-8)  (upper bound)

fig, ax = plt.subplots(figsize=(9, 6))

ax.plot(LL_range, A_line, "k-",  linewidth=1.5, label="A-line: PI = 0.73(LL−20)")
ax.plot(LL_range, U_line, "k--", linewidth=1.2, label="U-line: PI = 0.9(LL−8)", alpha=0.6)

# Vertical LL = 50 boundary
ax.axvline(50, color="gray", linestyle=":", linewidth=1.2)
ax.text(50, 55, "LL = 50", ha="center", fontsize=9, color="gray")

# Zone labels
ax.text(30, 5,  "ML / OL", fontsize=9, color="steelblue",  style="italic")
ax.text(65, 5,  "MH / OH", fontsize=9, color="steelblue",  style="italic")
ax.text(30, 20, "CL",       fontsize=9, color="firebrick",  style="italic")
ax.text(65, 35, "CH",       fontsize=9, color="firebrick",  style="italic")

# Plot the sample point
ax.scatter(LL, PI, s=100, color="red", zorder=5, label=f"Sample  (LL={LL}, PI={PI})")
ax.annotate(f"  ({LL}, {PI})", xy=(LL, PI), fontsize=9, color="red")

# A-line value at sample LL
A_at_LL = 0.73 * (LL - 20)
position = "above" if PI > A_at_LL else "below"
print(f"A-line value at LL={LL}: PI = {A_at_LL:.1f}")
print(f"Sample PI = {PI}  →  sample is {position} the A-line")

ax.set_xlim(0, 100)
ax.set_ylim(0, 60)
ax.set_xlabel("Liquid Limit (%)", fontsize=12)
ax.set_ylabel("Plasticity Index (%)", fontsize=12)
ax.set_title("Plasticity Chart (Casagrande)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig("plasticity_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → plasticity_chart.png")


## 7. USCS Soil Classification

Full classification logic following **ASTM D2487**:
- Coarse-grained if fines (% passing 0.075 mm) < 50 %
- Gravel if > 50 % of coarse fraction is retained on 4.75 mm sieve
- Sand otherwise
- Grading determined by Cu and Cc; plasticity by A-line position


In [ ]:
# ── Key percentages ────────────────────────────────────────────────────────────
pct_passing_075  = sieve.loc[sieve["Sieve Size (mm)"] == 0.075, "% Passing"].values[0]
pct_passing_475  = sieve.loc[sieve["Sieve Size (mm)"] == 4.75,  "% Passing"].values[0]
pct_retained_475 = 100 - pct_passing_475        # retained on #4 sieve
pct_coarse       = 100 - pct_passing_075         # coarse fraction
# Fraction of coarse retained on #4 = pct_retained_475 / pct_coarse
frac_gravel      = pct_retained_475 / pct_coarse if pct_coarse > 0 else 0

print(f"% passing 0.075 mm (fines) : {pct_passing_075:.1f} %")
print(f"% passing 4.75 mm          : {pct_passing_475:.1f} %")
print(f"Gravel fraction of coarse  : {frac_gravel*100:.1f} %")
print()

# ── A-line check ───────────────────────────────────────────────────────────────
above_A_line = PI > 0.73 * (LL - 20)

# ── USCS Decision Tree ─────────────────────────────────────────────────────────
if pct_passing_075 < 50:
    soil_type = "COARSE-GRAINED"

    if frac_gravel > 0.5:   # more than half of coarse fraction is gravel
        primary = "Gravel (G)"

        if pct_passing_075 < 5:
            if Cu >= 4 and 1 <= Cc <= 3:
                symbol, desc = "GW", "Well-Graded Gravel"
            else:
                symbol, desc = "GP", "Poorly-Graded Gravel"
        elif pct_passing_075 > 12:
            if above_A_line and PI >= 7:
                symbol, desc = "GC", "Clayey Gravel"
            else:
                symbol, desc = "GM", "Silty Gravel"
        else:
            # Borderline 5–12 %: dual symbol
            if Cu >= 4 and 1 <= Cc <= 3:
                base = "GW"
            else:
                base = "GP"
            if above_A_line and PI >= 4:
                symbol, desc = f"{base}-GC", f"{'Well' if base=='GW' else 'Poorly'}-Graded Gravel with Clay"
            else:
                symbol, desc = f"{base}-GM", f"{'Well' if base=='GW' else 'Poorly'}-Graded Gravel with Silt"

    else:
        primary = "Sand (S)"

        if pct_passing_075 < 5:
            if Cu >= 6 and 1 <= Cc <= 3:
                symbol, desc = "SW", "Well-Graded Sand"
            else:
                symbol, desc = "SP", "Poorly-Graded Sand"
        elif pct_passing_075 > 12:
            if above_A_line and PI >= 7:
                symbol, desc = "SC", "Clayey Sand"
            else:
                symbol, desc = "SM", "Silty Sand"
        else:
            if Cu >= 6 and 1 <= Cc <= 3:
                base = "SW"
            else:
                base = "SP"
            if above_A_line and PI >= 4:
                symbol, desc = f"{base}-SC", f"{'Well' if base=='SW' else 'Poorly'}-Graded Sand with Clay"
            else:
                symbol, desc = f"{base}-SM", f"{'Well' if base=='SW' else 'Poorly'}-Graded Sand with Silt"

else:
    soil_type = "FINE-GRAINED"
    primary   = "Silt / Clay"

    if LL < 50:     # low plasticity
        if above_A_line and PI >= 7:
            symbol, desc = "CL", "Lean Clay (low plasticity)"
        elif not above_A_line or PI < 4:
            symbol, desc = "ML", "Silt (low plasticity)"
        else:
            symbol, desc = "CL-ML", "Silty Clay (low plasticity)"
    else:           # high plasticity
        if above_A_line:
            symbol, desc = "CH", "Fat Clay (high plasticity)"
        else:
            symbol, desc = "MH", "Elastic Silt (high plasticity)"

# ── Print result ───────────────────────────────────────────────────────────────
print("=" * 50)
print("      USCS SOIL CLASSIFICATION RESULT")
print("=" * 50)
print(f"  Soil Type   : {soil_type}")
print(f"  Primary     : {primary}")
print(f"  USCS Symbol : {symbol}")
print(f"  Description : {desc}")
print("=" * 50)
print()
print("Key parameters used:")
print(f"  Fines (% passing 0.075 mm) = {pct_passing_075:.1f} %")
print(f"  Cu = {Cu:.2f}  |  Cc = {Cc:.2f}")
print(f"  LL = {LL}  |  PL = {PL}  |  PI = {PI}")
print(f"  Above A-line: {above_A_line}")


## 8. Full Results Summary

In [ ]:
summary = {
    "Total Mass (g)"          : total_mass,
    "% Fines (< 0.075 mm)"   : round(pct_passing_075, 2),
    "D10 (mm)"                : round(D10, 4),
    "D30 (mm)"                : round(D30, 4),
    "D60 (mm)"                : round(D60, 4),
    "Cu"                      : round(Cu, 3),
    "Cc"                      : round(Cc, 3),
    "Liquid Limit LL (%)"     : LL,
    "Plastic Limit PL (%)"    : PL,
    "Plasticity Index PI (%)" : PI,
    "USCS Symbol"             : symbol,
    "Classification"          : desc,
}

for k, v in summary.items():
    print(f"  {k:<30s}: {v}")
